In [155]:

import pandas as pd
from pandas_datareader import data
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV

In [156]:
df= pd.read_csv('HousingData.csv')

In [157]:
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,NaN,36.2


In [158]:
X = df.iloc[:,0:13].values
y = df.iloc[:,-1].values

In [159]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=4)

In [160]:
rt = DecisionTreeRegressor(criterion = 'squared_error', max_depth=5)

In [161]:
rt.fit(X_train,y_train)

DecisionTreeRegressor(max_depth=5)

In [162]:
y_pred = rt.predict(X_test)

In [163]:
r2_score(y_test,y_pred)

0.6511211906427594

In [164]:
from sklearn.model_selection import cross_val_score, KFold

In [165]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [166]:
cv_scores = cross_val_score(rt, X, y, cv=kf, scoring='r2')

print("CV R2 Scores for each fold:", cv_scores)
print("Mean CV R2 Score:          ", cv_scores.mean())
print("Standard Deviation:        ", cv_scores.std())

CV R2 Scores for each fold: [0.88689791 0.75701667 0.61240786 0.68887117 0.7914677 ]
Mean CV R2 Score:           0.7473322605960835
Standard Deviation:         0.09288981622831251


In [167]:
from sklearn.model_selection import GridSearchCV, KFold

# Parameter grid to search
param_grid = {
    'max_depth'        : [1,2, 3, 4, 5, 6, 7, 8, 9 , 10],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf' : [1, 2, 4, 8]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    DecisionTreeRegressor(),
    param_grid,
    cv=kf,
    scoring='r2',
    verbose=1
)

grid_search.fit(X, y)

print("Best Parameters:", grid_search.best_params_)
print("Best CV R2:     ", grid_search.best_score_)

Fitting 5 folds for each of 160 candidates, totalling 800 fits


Best Parameters: {'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 10}
Best CV R2:      0.7578914321157618


In [168]:
# Train best model on full training data
best_model = grid_search.best_estimator_

# Predict on test set
y_pred_tuned = best_model.predict(X_test)

print("Tuned Test R2:    ", r2_score(y_test, y_pred_tuned))
print("Tuned CV R2:      ", grid_search.best_score_)

Tuned Test R2:     0.9431955168226859
Tuned CV R2:       0.7578914321157618


## Hyperparameter Tuning

In [176]:
param_grid = {
    'max_depth':[2,4,8,10,None],
    'criterion':['msquared_error','absolute_error'],
    'max_features':[0.25,0.5,1.0],
    'min_samples_split':[0.25,0.5,1.0]
}
     

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('reg', DecisionTreeRegressor())
])

param_grid = {
    'reg__max_depth': [2, 4, 8, 10, None],
    'reg__criterion': ['squared_error', 'absolute_error'],
    'reg__max_features': [0.25, 0.5, 1.0],
    'reg__min_samples_split': [0.25, 0.5, 1.0]
}


reg = GridSearchCV(pipe, param_grid, cv=5)
reg.fit(X_train, y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('imputer', SimpleImputer()),
                                       ('reg', DecisionTreeRegressor())]),
             param_grid={'reg__max_depth': [2, 4, 6, 8, 10]})